# Global Pollution Analysis and Energy Recovery

## Aim
To find relationships between pollution levels and energy-related factors using the Apriori association rule mining algorithm.

### Work included
1. Load and clean `Global_Pollution_Analysis.csv`
2. Categorize pollution into Low, Medium and High
3. Use energy consumption per capita as a feature
4. Generate frequent itemsets using Apriori
5. Generate association rules using support and confidence
6. Check rules using lift and 5-fold rule validation
7. Plot frequent itemsets and association-rule networks


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import networkx as nx

pd.set_option("display.max_columns", None)


## 1. Loading the Dataset

The CSV file is kept in the same folder as the notebook.


In [ ]:
data = pd.read_csv("Global_Pollution_Analysis.csv")

data.columns = data.columns.str.strip()
data = data.drop_duplicates()

# Convert numerical columns to numbers
number_cols = data.select_dtypes(include=np.number).columns
data[number_cols] = data[number_cols].apply(pd.to_numeric, errors="coerce")

# Remove rows containing missing values
data = data.dropna().reset_index(drop=True)

print("Dataset shape:", data.shape)
data.head()


In [ ]:
print("Missing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())

print("\nData types:")
print(data.dtypes)


In [ ]:
# Basic statistics
data.describe().T


## 2. Pollution Severity and Feature Engineering

The pollution indexes are divided into three groups using the 33rd and 67th percentiles:

- Low
- Medium
- High

The dataset already contains `Energy_Consumption_Per_Capita (in MWh)`, so this variable is used directly. A Low/Medium/High category is also created for it so that it can be used in association rules.


In [ ]:
pollution_columns = {
    "Air_Pollution_Index": "Air_Severity",
    "Water_Pollution_Index": "Water_Severity",
    "Soil_Pollution_Index": "Soil_Severity"
}

for column, new_column in pollution_columns.items():
    low = data[column].quantile(1/3)
    high = data[column].quantile(2/3)

    data[new_column] = pd.cut(
        data[column],
        bins=[-np.inf, low, high, np.inf],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    ).astype(str)

# Energy per capita is already present in the original dataset
low = data["Energy_Consumption_Per_Capita (in MWh)"].quantile(1/3)
high = data["Energy_Consumption_Per_Capita (in MWh)"].quantile(2/3)

data["Energy_Per_Capita_Level"] = pd.cut(
    data["Energy_Consumption_Per_Capita (in MWh)"],
    bins=[-np.inf, low, high, np.inf],
    labels=["Low", "Medium", "High"],
    include_lowest=True
).astype(str)

# Simple high/low category for energy recovery
median_recovery = data["Energy_Recovered (in GWh)"].median()
data["Energy_Recovery_Level"] = np.where(
    data["Energy_Recovered (in GWh)"] >= median_recovery,
    "High", "Low"
)

data[[
    "Country", "Year",
    "Air_Severity", "Water_Severity", "Soil_Severity",
    "Energy_Consumption_Per_Capita (in MWh)",
    "Energy_Per_Capita_Level",
    "Energy_Recovery_Level"
]].head()


In [ ]:
# Pollution severity counts
severity_counts = pd.DataFrame({
    "Air": data["Air_Severity"].value_counts(),
    "Water": data["Water_Severity"].value_counts(),
    "Soil": data["Soil_Severity"].value_counts()
}).reindex(["Low", "Medium", "High"]).fillna(0)

severity_counts.plot(kind="bar", figsize=(8, 5))
plt.xlabel("Severity")
plt.ylabel("Number of records")
plt.title("Pollution Severity Categories")
plt.xticks(rotation=0)
plt.show()


## 3. Creating Transactions

Each row is treated as one transaction. The transaction contains the pollution severity and energy categories for that record.

Example:

`Air_High, Water_Medium, Soil_Low, Energy_Per_Capita_High, Energy_Recovery_Level_High`


In [ ]:
def make_transactions(data):
    columns = [
        "Air_Severity",
        "Water_Severity",
        "Soil_Severity",
        "Energy_Per_Capita_Level",
        "Energy_Recovery_Level"
    ]

    transactions = []

    for _, row in data.iterrows():
        items = set()

        for column in columns:
            if column.endswith("_Severity"):
                name = column.replace("_Severity", "")
            else:
                name = column

            items.add(f"{name}_{row[column]}")

        transactions.append(frozenset(items))

    return transactions

transactions = make_transactions(data)

print("Number of transactions:", len(transactions))
print("First transaction:", transactions[0])


## 4. Apriori Algorithm

The following functions implement Apriori directly instead of depending on a library.

The algorithm:
1. Finds frequent 1-itemsets.
2. Joins them to create larger candidates.
3. Removes candidates whose subsets are not frequent.
4. Keeps itemsets whose support is above the minimum support.


In [ ]:
def get_support(itemset, transactions):
    count = 0

    for transaction in transactions:
        if itemset.issubset(transaction):
            count += 1

    return count / len(transactions)


def generate_candidates(previous_itemsets, k):
    previous_itemsets = list(previous_itemsets)
    candidates = set()

    for i in range(len(previous_itemsets)):
        for j in range(i + 1, len(previous_itemsets)):
            candidate = previous_itemsets[i] | previous_itemsets[j]

            if len(candidate) != k:
                continue

            # Apriori pruning
            valid = True
            for subset in itertools.combinations(candidate, k - 1):
                if frozenset(subset) not in previous_itemsets:
                    valid = False
                    break

            if valid:
                candidates.add(frozenset(candidate))

    return candidates


def apriori(transactions, min_support=0.08):
    # First level
    item_counts = {}

    for transaction in transactions:
        for item in transaction:
            itemset = frozenset([item])
            item_counts[itemset] = item_counts.get(itemset, 0) + 1

    current = {
        itemset: count / len(transactions)
        for itemset, count in item_counts.items()
        if count / len(transactions) >= min_support
    }

    all_itemsets = dict(current)
    k = 2

    while current:
        candidates = generate_candidates(current.keys(), k)
        next_level = {}

        for candidate in candidates:
            candidate_support = get_support(candidate, transactions)

            if candidate_support >= min_support:
                next_level[candidate] = candidate_support

        all_itemsets.update(next_level)
        current = next_level
        k += 1

    return all_itemsets


min_support = 0.08
frequent_itemsets = apriori(transactions, min_support)

print("Number of frequent itemsets:", len(frequent_itemsets))


In [ ]:
frequent_table = pd.DataFrame([
    {
        "Itemset": ", ".join(sorted(itemset)),
        "Support": support_value
    }
    for itemset, support_value in frequent_itemsets.items()
]).sort_values("Support", ascending=False)

frequent_table.head(15)


In [ ]:
# Frequent itemset visualization
top_items = frequent_table.head(12).sort_values("Support")

plt.figure(figsize=(9, 6))
plt.barh(top_items["Itemset"], top_items["Support"])
plt.xlabel("Support")
plt.title("Top Frequent Itemsets")
plt.tight_layout()
plt.show()


## 5. Association Rule Mining

A rule has the form:

**A → B**

The important measures are:

- **Support:** how often A and B occur together.
- **Confidence:** how often B occurs when A occurs.
- **Lift:** compares the rule with what would be expected if A and B were independent.

Minimum support and confidence are used to remove weak rules.


In [ ]:
def make_rules(frequent_itemsets, transactions, min_confidence=0.60):
    rules = []

    for itemset, itemset_support in frequent_itemsets.items():

        if len(itemset) < 2:
            continue

        items = list(itemset)

        for size in range(1, len(items)):
            for part in itertools.combinations(items, size):

                antecedent = frozenset(part)
                consequent = itemset - antecedent

                antecedent_support = frequent_itemsets.get(antecedent)

                if antecedent_support is None or antecedent_support == 0:
                    continue

                consequent_support = frequent_itemsets.get(consequent)

                if consequent_support is None:
                    consequent_support = get_support(
                        consequent, transactions
                    )

                confidence = itemset_support / antecedent_support

                if consequent_support == 0:
                    continue

                lift = confidence / consequent_support

                if confidence >= min_confidence:
                    rules.append({
                        "Antecedent": ", ".join(sorted(antecedent)),
                        "Consequent": ", ".join(sorted(consequent)),
                        "Support": itemset_support,
                        "Confidence": confidence,
                        "Lift": lift
                    })

    return pd.DataFrame(rules).sort_values(
        ["Lift", "Confidence"],
        ascending=False
    ).reset_index(drop=True)


min_confidence = 0.60
rules = make_rules(
    frequent_itemsets,
    transactions,
    min_confidence
)

rules.head(15)


In [ ]:
# Rules involving pollution and energy
energy_pollution_rules = rules[
    (
        rules["Antecedent"].str.contains("Air_|Water_|Soil_", regex=True) |
        rules["Consequent"].str.contains("Air_|Water_|Soil_", regex=True)
    ) &
    (
        rules["Antecedent"].str.contains("Energy_", regex=True) |
        rules["Consequent"].str.contains("Energy_", regex=True)
    )
]

energy_pollution_rules.head(15)


In [ ]:
# Plot the strongest rules by lift
top_rules = rules.head(10).copy()
top_rules["Rule"] = (
    top_rules["Antecedent"] + " -> " + top_rules["Consequent"]
)

top_rules = top_rules.sort_values("Lift")

plt.figure(figsize=(9, 6))
plt.barh(top_rules["Rule"], top_rules["Lift"])
plt.xlabel("Lift")
plt.title("Top Association Rules")
plt.tight_layout()
plt.show()


## 6. Rule Cross-Validation

To check whether the rules are stable, the transactions are divided into five folds.

The rules are generated from the full dataset using the selected thresholds. Each rule is then checked on each held-out fold. Mean support, confidence and lift across the folds are reported.

This is a stability check for association rules rather than the same type of cross-validation used for a prediction model.


In [ ]:
def evaluate_rule(rule, test_transactions):
    antecedent = frozenset(
        x.strip() for x in rule["Antecedent"].split(",")
    )
    consequent = frozenset(
        x.strip() for x in rule["Consequent"].split(",")
    )

    both = antecedent | consequent

    ant_support = get_support(antecedent, test_transactions)
    cons_support = get_support(consequent, test_transactions)
    both_support = get_support(both, test_transactions)

    if ant_support == 0:
        confidence = np.nan
    else:
        confidence = both_support / ant_support

    if cons_support == 0 or np.isnan(confidence):
        lift = np.nan
    else:
        lift = confidence / cons_support

    return both_support, confidence, lift


rng = np.random.RandomState(42)
indices = np.arange(len(transactions))
rng.shuffle(indices)

folds = np.array_split(indices, 5)

cv_rows = []

for _, rule in rules.head(20).iterrows():

    fold_values = []

    for fold in folds:
        test_transactions = [transactions[i] for i in fold]
        fold_values.append(
            evaluate_rule(rule, test_transactions)
        )

    values = np.array(fold_values)

    cv_rows.append({
        "Antecedent": rule["Antecedent"],
        "Consequent": rule["Consequent"],
        "Mean Support": np.nanmean(values[:, 0]),
        "Mean Confidence": np.nanmean(values[:, 1]),
        "Mean Lift": np.nanmean(values[:, 2]),
        "Valid Folds": np.sum(~np.isnan(values[:, 1]))
    })

cv_rules = pd.DataFrame(cv_rows)

cv_rules.sort_values(
    ["Mean Lift", "Mean Confidence"],
    ascending=False
).head(15)


## 7. Association Rule Network

The network shows the direction of the strongest rules. Each arrow represents:

**Antecedent → Consequent**

The network is useful for quickly seeing which pollution and energy categories appear together in the discovered rules.


In [ ]:
G = nx.DiGraph()

for _, rule in rules.head(12).iterrows():

    antecedent = rule["Antecedent"]
    consequent = rule["Consequent"]

    G.add_node(antecedent)
    G.add_node(consequent)

    G.add_edge(
        antecedent,
        consequent,
        lift=rule["Lift"]
    )

plt.figure(figsize=(11, 7))

pos = nx.spring_layout(G, seed=42)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=1700
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=7
)

nx.draw_networkx_edges(
    G,
    pos,
    arrows=True,
    arrowsize=15
)

plt.title("Association Rule Network")
plt.axis("off")
plt.show()


## 8. Conclusion

Apriori was used to find frequent combinations of pollution severity and energy-related categories. Association rules were then generated using minimum support and confidence.

Support, confidence and lift were used to evaluate the rules. A five-fold rule validation was also carried out to check whether the discovered relationships remain reasonably stable across different parts of the dataset.

The results can be used to identify pollution conditions that frequently occur together with particular energy-consumption or energy-recovery levels. These are associations in the dataset and should not be treated as proof of cause and effect.
